In [5]:
!pip -q install streamlit pyjwt bcrypt pyngrok

In [6]:
%%writefile db.py
import os, sqlite3, bcrypt, datetime, time, re
DB_NAME = os.getenv('DB_NAME', 'milestone1_users.db')
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS = 300

def connect():
    conn = sqlite3.connect(DB_NAME, timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute('PRAGMA foreign_keys = ON')
    return conn

def now_text():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')

def hash_text(value):
    return bcrypt.hashpw(value.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')

def verify_hash(value, hashed):
    try: return bool(hashed) and bcrypt.checkpw(value.encode('utf-8'), hashed.encode('utf-8') if isinstance(hashed, str) else hashed)
    except (ValueError, TypeError): return False

def init_db():
    with connect() as c:
        c.execute('''CREATE TABLE IF NOT EXISTS users (
            email TEXT PRIMARY KEY COLLATE NOCASE,
            username TEXT NOT NULL UNIQUE COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            role TEXT NOT NULL DEFAULT 'user',
            created_at TEXT NOT NULL
        )''')
        c.execute('''CREATE TABLE IF NOT EXISTS password_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT NOT NULL COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            set_at TEXT NOT NULL,
            FOREIGN KEY(email) REFERENCES users(email) ON DELETE CASCADE
        )''')
        c.execute('''CREATE TABLE IF NOT EXISTS login_attempts (
            email TEXT PRIMARY KEY COLLATE NOCASE, attempts INTEGER NOT NULL, last_attempt REAL NOT NULL
        )''')

def valid_password(password):
    return (len(password) >= 8 and re.search(r'[A-Z]', password) and re.search(r'[a-z]', password)
            and re.search(r'\d', password) and re.search(r'[^A-Za-z0-9]', password))

def register_user(username, email, password, question, answer):
    email, username = email.strip().lower(), username.strip()
    hashed = hash_text(password); stamp = now_text()
    try:
        with connect() as c:
            c.execute('INSERT INTO users(username,email,password_hash,security_question,security_answer_hash,role,created_at) VALUES(?,?,?,?,?,?,?)',
                      (username, email, hashed, question, hash_text(answer.strip().casefold()), 'user', stamp))
            c.execute('INSERT INTO password_history(email,password_hash,set_at) VALUES(?,?,?)', (email, hashed, stamp))
        return True, 'Account created successfully.'
    except sqlite3.IntegrityError:
        return False, 'That username or email is already registered.'

def get_user(email):
    with connect() as c: return c.execute('SELECT * FROM users WHERE email=? COLLATE NOCASE', (email.strip(),)).fetchone()

def username_exists(username):
    with connect() as c: return c.execute('SELECT 1 FROM users WHERE username=? COLLATE NOCASE', (username.strip(),)).fetchone() is not None

def email_exists(email): return get_user(email) is not None

def get_security_question(email):
    with connect() as c: row = c.execute('SELECT security_question FROM users WHERE email=? COLLATE NOCASE', (email.strip(),)).fetchone()
    return row['security_question'] if row else None

def verify_security_answer(email, answer):
    with connect() as c: row = c.execute('SELECT security_answer_hash FROM users WHERE email=? COLLATE NOCASE', (email.strip(),)).fetchone()
    return bool(row and verify_hash(answer.strip().casefold(), row['security_answer_hash']))

def login_status(email):
    with connect() as c: row = c.execute('SELECT attempts,last_attempt FROM login_attempts WHERE email=? COLLATE NOCASE', (email.strip(),)).fetchone()
    if not row: return False, 0
    remaining = LOCKOUT_SECONDS - (time.time() - row['last_attempt'])
    if row['attempts'] >= MAX_LOGIN_ATTEMPTS and remaining > 0: return True, int(remaining)
    if remaining <= 0: clear_login_attempts(email)
    return False, 0

def record_failed_login(email):
    with connect() as c:
        row = c.execute('SELECT attempts FROM login_attempts WHERE email=? COLLATE NOCASE', (email.strip(),)).fetchone()
        attempts = (row['attempts'] if row else 0) + 1
        c.execute('INSERT OR REPLACE INTO login_attempts(email,attempts,last_attempt) VALUES(?,?,?)', (email.strip().lower(), attempts, time.time()))

def clear_login_attempts(email):
    with connect() as c: c.execute('DELETE FROM login_attempts WHERE email=? COLLATE NOCASE', (email.strip(),))

def authenticate(email, password):
    locked, seconds = login_status(email)
    if locked: return False, f'Too many attempts. Try again in about {max(1, seconds)} seconds.'
    user = get_user(email)
    if user and verify_hash(password, user['password_hash']):
        clear_login_attempts(email); return True, user
    record_failed_login(email)
    return False, 'Email or password is incorrect.'

def password_reused(email, new_password):
    with connect() as c: rows = c.execute('SELECT password_hash FROM password_history WHERE email=? COLLATE NOCASE ORDER BY id DESC LIMIT 5', (email.strip(),)).fetchall()
    return any(verify_hash(new_password, r['password_hash']) for r in rows)

def update_password(email, new_password):
    hashed, stamp = hash_text(new_password), now_text()
    with connect() as c:
        c.execute('UPDATE users SET password_hash=? WHERE email=? COLLATE NOCASE', (hashed, email.strip()))
        c.execute('INSERT INTO password_history(email,password_hash,set_at) VALUES(?,?,?)', (email.strip().lower(), hashed, stamp))
        c.execute('DELETE FROM login_attempts WHERE email=? COLLATE NOCASE', (email.strip(),))

def list_users():
    with connect() as c: return c.execute("SELECT username,email,role,created_at FROM users WHERE role!='admin' ORDER BY created_at DESC").fetchall()

def ensure_admin(email, password):
    # The milestone requires a separate admin login. Keep credentials in Colab Secrets where possible.
    if not email or not password: return
    with connect() as c:
        row = c.execute('SELECT email FROM users WHERE email=? COLLATE NOCASE', (email,)).fetchone()
        if row:
            c.execute("UPDATE users SET role='admin' WHERE email=? COLLATE NOCASE", (email,))
        else:
            stamp, hashed = now_text(), hash_text(password)
            c.execute('INSERT INTO users(username,email,password_hash,security_question,security_answer_hash,role,created_at) VALUES(?,?,?,?,?,?,?)',
                      ('Administrator', email.lower(), hashed, 'Admin account', hash_text('not-used'), 'admin', stamp))
            c.execute('INSERT INTO password_history(email,password_hash,set_at) VALUES(?,?,?)', (email.lower(), hashed, stamp))


Overwriting db.py


In [7]:
%%writefile app.py
import os, re, time, secrets, smtplib, datetime
from email.mime.text import MIMEText
import jwt
import streamlit as st
import db

st.set_page_config(page_title='Maritime Freight Intelligence | Milestone 1', page_icon='⚓', layout='centered', initial_sidebar_state='collapsed')
db.init_db()
JWT_SECRET = os.getenv('JWT_SECRET', '')
SENDER_EMAIL = os.getenv('EMAIL_ADDRESS', '')
EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD', '')
ADMIN_EMAIL = os.getenv('ADMIN_EMAIL', 'admin@llm.com').strip().lower()
ADMIN_PASSWORD = os.getenv('ADMIN_PASSWORD', '')
db.ensure_admin(ADMIN_EMAIL, ADMIN_PASSWORD)
st.markdown('''<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Manrope:wght@500;600;700;800&display=swap');
:root{--ink:#102a43;--muted:#627d98;--brand:#087e8b;--brand2:#066773;--line:#d9e6ec;--surface:#ffffff;--navy:#082f49;--teal:#10b8a6;}
.stApp{background:radial-gradient(ellipse at 8% 4%,#d9f3f1 0,transparent 32%),radial-gradient(ellipse at 94% 90%,#dcecf8 0,transparent 30%),linear-gradient(135deg,#f5fafc,#edf5f8);color:var(--ink);font-family:'DM Sans',sans-serif;}
[data-testid="stHeader"]{background:transparent}.block-container{max-width:850px;padding-top:1.8rem;padding-bottom:3rem}
h1,h2,h3{font-family:'Manrope',sans-serif!important;color:var(--ink)!important;letter-spacing:-.035em}
.brand-mark{display:inline-flex;align-items:center;justify-content:center;width:54px;height:54px;border-radius:17px;background:linear-gradient(145deg,#082f49,#087e8b 72%,#10b8a6);color:white;font-size:27px;box-shadow:0 10px 24px #082f4930;border:1px solid #ffffff55}
.project-ribbon{display:flex;align-items:center;gap:.55rem;color:#42657a;font-size:.76rem;font-weight:800;letter-spacing:.12em;text-transform:uppercase;margin:.8rem 0 .25rem}
.project-ribbon .dot{width:7px;height:7px;background:#10b8a6;border-radius:50%;box-shadow:0 0 0 4px #10b8a61f}
.eyebrow{color:#087e8b;text-transform:uppercase;letter-spacing:.15em;font-size:.72rem;font-weight:800;margin-top:.7rem}
.hero-title{font-family:'Manrope',sans-serif;font-size:2.15rem;font-weight:800;line-height:1.15;margin:.35rem 0 .55rem;color:#102a43}
.hero-sub{color:#627d98;font-size:1rem;line-height:1.65;margin-bottom:1.5rem;max-width:680px}
[data-testid="stForm"],[data-testid="stVerticalBlockBorderWrapper"]{background:rgba(255,255,255,.94);border:1px solid var(--line);border-radius:20px;padding:1.25rem;box-shadow:0 18px 46px rgba(8,47,73,.08)}
/* Explicit widget colors prevent low-contrast or invisible text. */
[data-testid="stTextInput"] label,[data-testid="stSelectbox"] label,[data-testid="stRadio"] label,[data-testid="stCheckbox"] label{font-weight:650;color:#23445b!important}
[data-testid="stTextInput"] input,[data-testid="stTextArea"] textarea,[data-testid="stNumberInput"] input,[data-testid="stDateInput"] input,[data-testid="stTimeInput"] input{border-radius:11px;border:1px solid #c9dbe4;min-height:44px;background:#fbfdfe!important;color:#102a43!important;-webkit-text-fill-color:#102a43!important;caret-color:#087e8b}
[data-testid="stTextInput"] input::placeholder,[data-testid="stTextArea"] textarea::placeholder{color:#627d98!important;-webkit-text-fill-color:#627d98!important;opacity:1}
[data-testid="stTextInput"] input:focus,[data-testid="stTextArea"] textarea:focus{border-color:#087e8b;box-shadow:0 0 0 3px #087e8b1c}
/* Keep Chrome saved-email/password autofill from tinting the field grey. */
[data-testid="stTextInput"] input:-webkit-autofill,[data-testid="stTextInput"] input:-webkit-autofill:hover,[data-testid="stTextInput"] input:-webkit-autofill:focus,[data-testid="stTextInput"] input:autofill,[data-testid="stTextInput"] input:autofill:hover,[data-testid="stTextInput"] input:autofill:focus{-webkit-text-fill-color:#102a43!important;caret-color:#087e8b!important;-webkit-box-shadow:0 0 0 1000px #fbfdfe inset!important;box-shadow:0 0 0 1000px #fbfdfe inset!important;border:1px solid #c9dbe4!important;transition:background-color 999999s ease-out 0s}
[data-testid="stRadio"] [role="radiogroup"]{gap:.6rem}
[data-testid="stRadio"] [role="radiogroup"] label,[data-testid="stRadio"] [role="radiogroup"] label p,[data-testid="stRadio"] [role="radiogroup"] label span{color:#23445b!important;opacity:1!important}
[data-testid="stSelectbox"] div[data-baseweb="select"]>div{background:#fbfdfe!important;border-color:#c9dbe4!important;color:#102a43!important}
[data-baseweb="popover"] [role="listbox"],[data-baseweb="menu"],[data-baseweb="menu"] li{background:#fff!important;color:#102a43!important}
[data-baseweb="popover"] [role="option"],[data-baseweb="popover"] [role="option"] *{color:#102a43!important}
.stButton>button,.stFormSubmitButton>button{border-radius:11px;min-height:44px;font-weight:700;border:1px solid #087e8b;background:linear-gradient(135deg,#087e8b,#066773);color:white!important;transition:all .15s ease}
.stButton>button p,.stFormSubmitButton>button p{color:white!important}
.stButton>button:hover,.stFormSubmitButton>button:hover{background:#055964;border-color:#055964;color:white!important;transform:translateY(-1px)}
[data-testid="stAlert"]{border-radius:12px}.small-note{color:#627d98;font-size:.84rem;text-align:center;margin-top:1rem}
[data-testid="stCaptionContainer"],[data-testid="stCaptionContainer"] p{color:#526d82!important}
/* FIX: Improve metric card text visibility on the light dashboard */
div[data-testid="stMetric"] {
    background: linear-gradient(145deg, #ffffff, #f2fafb) !important;
    border: 1px solid #d9e6ec !important;
    padding: 1rem !important;
    border-radius: 15px !important;
    color: #102a43 !important;
}

/* Force readable colors on every text element inside metric cards */
div[data-testid="stMetric"] * {
    opacity: 1 !important;
}

/* Metric labels */
div[data-testid="stMetric"] label,
div[data-testid="stMetric"] label *,
div[data-testid="stMetric"] [data-testid="stMetricLabel"],
div[data-testid="stMetric"] [data-testid="stMetricLabel"] * {
    color: #526d82 !important;
    -webkit-text-fill-color: #526d82 !important;
    font-weight: 600 !important;
}

/* Metric values, including nested Streamlit elements */
div[data-testid="stMetric"] [data-testid="stMetricValue"],
div[data-testid="stMetric"] [data-testid="stMetricValue"] *,
div[data-testid="stMetric"] [data-testid="stMetricDelta"],
div[data-testid="stMetric"] [data-testid="stMetricDelta"] * {
    color: #102a43 !important;
    -webkit-text-fill-color: #102a43 !important;
    opacity: 1 !important;
}
[data-testid="stDataFrame"]{border:1px solid var(--line);border-radius:12px;overflow:hidden}
[data-testid="stRadio"] [role="radiogroup"]{gap:.6rem}
footer{visibility:hidden}#MainMenu{visibility:hidden}
@media(max-width:640px){.block-container{padding:1.1rem .8rem}.hero-title{font-size:1.7rem}.hero-sub{font-size:.94rem}}
</style>''', unsafe_allow_html=True)

QUESTIONS = ['What was the name of your first school?', 'What is the name of your hometown?', 'What was your childhood nickname?', 'What is the name of your favorite teacher?']
def init_state():
    defaults = {'page':'Login','login_email':'','reset_email':'','recovery_method':'Security question','otp_hash':None,'otp_expiry':None,'otp_attempts':0,'reset_verified':False,'session_token':None,'logged_email':None,'logged_role':None,'admin_mode':False}
    for k,v in defaults.items(): st.session_state.setdefault(k,v)
init_state()

def make_session_token(user):
    if not JWT_SECRET: return None
    now = int(time.time())
    return jwt.encode({'sub':user['email'],'role':user['role'],'iat':now,'exp':now+7200}, JWT_SECRET, algorithm='HS256')

def current_user():
    token=st.session_state.session_token
    if not token or not JWT_SECRET: return None
    try:
        payload=jwt.decode(token,JWT_SECRET,algorithms=['HS256'])
        user=db.get_user(payload['sub'])
        if not user or user['role'] != payload.get('role'): return None
        return user
    except jwt.PyJWTError: return None

def go(page): st.session_state.page=page; st.rerun()
def sign_out():
    for key in ['session_token','logged_email','logged_role','reset_verified','otp_hash','otp_expiry']:
        st.session_state[key]=None
    st.session_state.admin_mode=False; st.session_state.page='Login'; st.rerun()
def password_rules(p):
    return (len(p)>=8 and re.search(r'[A-Z]',p) and re.search(r'[a-z]',p) and re.search(r'\d',p) and re.search(r'[^A-Za-z0-9]',p))
def email_valid(e): return bool(re.fullmatch(r'[^\s@]{2,}@[^\s@]{2,}\.[A-Za-z]{2,}',e.strip()))
def format_ist(value):
    """Convert stored UTC ISO timestamps to a readable India Standard Time string."""
    if not value:
        return '—'
    try:
        parsed = datetime.datetime.fromisoformat(str(value).strip().replace('Z', '+00:00'))
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=datetime.timezone.utc)
        ist = parsed.astimezone(datetime.timezone(datetime.timedelta(hours=5, minutes=30)))
        return ist.strftime('%d %b %Y, %I:%M:%S %p IST')
    except (ValueError, TypeError, OverflowError):
        return str(value)
def header(kicker,title,subtitle):
    st.markdown('<div class="brand-mark">⚓</div><div class="project-ribbon"><span class="dot"></span> Agentic AI for Maritime Freight Pricing and Route Optimization</div>',unsafe_allow_html=True)
    st.markdown(f'<div class="eyebrow">{kicker}</div><div class="hero-title">{title}</div><div class="hero-sub">{subtitle}</div>',unsafe_allow_html=True)
def send_otp(email, otp):
    if not (SENDER_EMAIL and EMAIL_PASSWORD): return False, 'Email configuration is missing. Add EMAIL_ADDRESS and EMAIL_PASSWORD to Colab Secrets.'
    msg=MIMEText(f'Your password reset verification code is {otp}. It expires in 5 minutes. If you did not request this, you can ignore this email.')
    msg['Subject']='Your password reset verification code'; msg['From']=SENDER_EMAIL; msg['To']=email
    try:
        with smtplib.SMTP_SSL('smtp.gmail.com',465,timeout=20) as server:
            server.login(SENDER_EMAIL,EMAIL_PASSWORD); server.send_message(msg)
        return True,'Verification code sent.'
    except Exception:
        return False,'Unable to send the email. Check the sender address and Gmail App Password.'
def issue_otp(email):
    otp=f'{secrets.randbelow(900000)+100000}'
    st.session_state.otp_hash=db.hash_text(otp); st.session_state.otp_expiry=time.time()+300; st.session_state.otp_attempts=0
    return send_otp(email,otp)
def verify_otp(value):
    if not st.session_state.otp_expiry or time.time()>st.session_state.otp_expiry: return False,'The code has expired. Request a new code.'
    if st.session_state.otp_attempts>=5: return False,'Too many attempts. Request a new code.'
    st.session_state.otp_attempts+=1
    if db.verify_hash(value,st.session_state.otp_hash):
        st.session_state.reset_verified=True; st.session_state.otp_hash=None; return True,'Email verified.'
    return False,'Incorrect verification code.'

user=current_user()
if st.session_state.session_token and not user: st.session_state.session_token=None
if user:
    if user['role']=='admin':
        header('FLEET ACCESS CONTROL','Administrator console','Review registered accounts and access status for the maritime project workspace.')
        users=db.list_users()
        c1,c2=st.columns(2); c1.metric('Registered users',len(users)); c2.metric('Access level','Administrator')
        if users:
            st.dataframe([{'Username':r['username'],'Email':r['email'],'Role':r['role'].title(),'Registered (IST)':format_ist(r['created_at'])} for r in users],use_container_width=True,hide_index=True)
        else: st.info('No registered users yet. New accounts will appear here.')
        if st.button('Log out',use_container_width=True): sign_out()
    else:
        header('MARITIME WORKSPACE','Welcome aboard, '+user['username'],'Your secure project workspace is ready. Freight pricing and route-optimization tools will be introduced in later milestones.')
        with st.container(border=True):
            st.markdown('#### Workspace access overview')
            st.write('**Username** · '+user['username']); st.write('**Email** · '+user['email']); st.write('**Member since** · '+format_ist(user['created_at']))
            st.success('Secure session active · Maritime project workspace access granted.')
        if st.button('Log out',use_container_width=True): sign_out()
else:
    if not JWT_SECRET: st.warning('Setup needed: add a strong JWT_SECRET to Colab Secrets before using authentication.')
    if st.session_state.page=='Login':
        header('PORTAL ACCESS','Sign in to continue','Access the maritime freight intelligence project workspace with your registered account.')
        with st.form('login_form'):
            email=st.text_input('Email address',placeholder='you@example.com').strip().lower()
            password=st.text_input('Password',type='password',placeholder='Enter your password')
            submitted=st.form_submit_button('Sign in →',use_container_width=True)
        if submitted:
            if not email or not password: st.error('Please fill in both fields.')
            elif not JWT_SECRET: st.error('JWT_SECRET is not configured in Colab Secrets.')
            else:
                ok,result=db.authenticate(email,password)
                if ok and result['role'] == 'admin':
                    st.error('Use the separate Administrator sign-in option for this account.')
                elif ok:
                    token=make_session_token(result)
                    if token:
                        st.session_state.session_token=token; st.session_state.logged_email=result['email']; st.rerun()
                    else: st.error('Could not create a secure session. Check JWT_SECRET.')
                else: st.error(result)
        a,b=st.columns(2)
        with a:
            if st.button('Create an account',use_container_width=True): go('Signup')
        with b:
            if st.button('Forgot password?',use_container_width=True):
                st.session_state.reset_email=''; st.session_state.recovery_method='Security question'; st.session_state.reset_verified=False; go('Forgot Password')
        st.markdown('<div class="small-note">⚓ Maritime Freight Intelligence · Protected access · Secure sessions</div>',unsafe_allow_html=True)
    elif st.session_state.page=='Signup':
        header('NEW CREW MEMBER','Create your account','Register to access the maritime project workspace.')
        with st.form('signup_form'):
            username=st.text_input('Username',placeholder='Choose a username').strip()
            email=st.text_input('Email address',placeholder='you@example.com').strip().lower()
            password=st.text_input('Password',type='password',placeholder='Create a strong password')
            confirm=st.text_input('Confirm password',type='password',placeholder='Re-enter your password')
            question=st.selectbox('Security question',QUESTIONS)
            answer=st.text_input('Security answer',placeholder='Your answer').strip()
            st.caption('Password must contain 8+ characters, uppercase and lowercase letters, a number, and a special character.')
            submitted=st.form_submit_button('Create account →',use_container_width=True)
        if submitted:
            if not all([username,email,password,confirm,answer]): st.error('All fields are mandatory.')
            elif len(username)<3: st.error('Username must contain at least 3 characters.')
            elif not email_valid(email): st.error('Enter a valid email address.')
            elif not password_rules(password): st.error('Password does not meet all the requirements shown above.')
            elif password!=confirm: st.error('Passwords do not match.')
            elif not JWT_SECRET: st.error('JWT_SECRET is not configured in Colab Secrets.')
            else:
                ok,msg=db.register_user(username,email,password,question,answer)
                if ok: st.success(msg+' You can now sign in.'); st.session_state.page='Login'
                else: st.error(msg)
        if st.button('← Back to login'): go('Login')
    elif st.session_state.page=='Forgot Password':
        header('ACCOUNT RECOVERY','Reset your password','Verify your identity using your security question or a one-time email code.')
        email=st.text_input('Registered email address',value=st.session_state.reset_email,placeholder='you@example.com').strip().lower()
        method=st.radio('Choose recovery method',['Security question','Email OTP'],horizontal=True,index=0 if st.session_state.recovery_method=='Security question' else 1)
        st.session_state.recovery_method=method
        if method=='Security question':
            question=db.get_security_question(email) if email and email_valid(email) else None
            if question: st.info('Security question: '+question)
            answer=st.text_input('Your security answer',placeholder='Enter the answer you set during signup')
            if st.button('Verify answer →',use_container_width=True):
                if not email or not email_valid(email): st.error('Enter a valid registered email address.')
                elif not question: st.error('Unable to verify these details. Check the email and try again.')
                elif not db.verify_security_answer(email,answer): st.error('Unable to verify these details. Check your answer and try again.')
                else: st.session_state.reset_email=email; st.session_state.reset_verified=True; st.success('Identity verified. You can set a new password below.')
        else:
            if st.button('Send email OTP →',use_container_width=True):
                if not email or not email_valid(email): st.error('Enter a valid registered email address.')
                elif not db.email_exists(email): st.error('If the account exists, a verification email can be requested. Check the address and try again.')
                else:
                    ok,msg=issue_otp(email)
                    if ok: st.session_state.reset_email=email; st.success(msg)
                    else: st.error(msg)
            if st.session_state.otp_hash and st.session_state.reset_email==email:
                otp=st.text_input('Six-digit verification code',max_chars=6,placeholder='000000')
                if st.button('Verify code',use_container_width=True):
                    if not re.fullmatch(r'\d{6}',otp): st.error('Enter the six-digit code from your email.')
                    else:
                        ok,msg=verify_otp(otp)
                        if ok: st.success(msg)
                        else: st.error(msg)
        if st.session_state.reset_verified and st.session_state.reset_email==email:
            st.divider(); st.markdown('#### Choose a new password')
            new=st.text_input('New password',type='password',key='new_password')
            confirm=st.text_input('Confirm new password',type='password',key='confirm_new_password')
            st.caption('Use at least 8 characters with uppercase, lowercase, a number and a special character.')
            if st.button('Update password →',use_container_width=True):
                if not password_rules(new): st.error('Password does not meet all requirements.')
                elif new!=confirm: st.error('Passwords do not match.')
                elif db.password_reused(email,new): st.error('Choose a password you have not used recently.')
                else:
                    db.update_password(email,new); st.success('Password updated. Please sign in with your new password.')
                    st.session_state.reset_verified=False; st.session_state.otp_hash=None; st.session_state.otp_expiry=None; st.session_state.page='Login'
        if st.button('← Back to login'): go('Login')
    elif st.session_state.page=='Admin Login':
        header('CONTROL TOWER','Administrator sign in','Restricted access for authorized project administrators only.')
        with st.form('admin_login_form'):
            email=st.text_input('Admin email',placeholder='admin@example.com').strip().lower()
            password=st.text_input('Admin password',type='password')
            submitted=st.form_submit_button('Sign in as admin →',use_container_width=True)
        if submitted:
            if not ADMIN_PASSWORD: st.error('Set ADMIN_PASSWORD in Colab Secrets first.')
            elif email==ADMIN_EMAIL and secrets.compare_digest(password,ADMIN_PASSWORD) and JWT_SECRET:
                user=db.get_user(ADMIN_EMAIL); token=make_session_token(user)
                if token: st.session_state.session_token=token; st.rerun()
                else: st.error('Could not create admin session.')
            else: st.error('Admin email or password is incorrect.')
        if st.button('← Back to login'): go('Login')

if not user and st.session_state.page=='Login':
    st.markdown('<div class="small-note">Project administrator? Use the separate control-tower sign-in.</div>',unsafe_allow_html=True)
    if st.button('Administrator sign in',use_container_width=True): go('Admin Login')



Overwriting app.py


In [8]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

required = ['JWT_SECRET', 'EMAIL_ADDRESS', 'EMAIL_PASSWORD', 'NGROK_AUTHTOKEN', 'ADMIN_EMAIL', 'ADMIN_PASSWORD']
missing = []
for name in required:
    try:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
        else:
            missing.append(name)
    except Exception:
        missing.append(name)
if missing:
    raise RuntimeError('Add these values in Colab Secrets and enable notebook access: ' + ', '.join(missing))

# Run this launch cell once per Colab runtime. If port 8501 is already occupied, restart the runtime first.
time.sleep(2)
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'], env=os.environ.copy())
time.sleep(4)
try:
    tunnel = ngrok.connect(8501, 'http')
    print('Open the app using this URL:', tunnel.public_url)
    print('Keep this cell running while you use the app.')
except Exception as e:
    process.terminate()
    raise RuntimeError('Could not start ngrok. Verify NGROK_AUTHTOKEN and try again.') from e


Open the app using this URL: https://92a5-34-23-164-28.ngrok-free.app
Keep this cell running while you use the app.
